In [1]:
"""
Test strategies based on simple technical indicators (RSI, MA, MACD, Bollinger etc.) on
    - random processes with length of 1000 candles for example 
    - (preferred) (if we think the trading process is not really random walk) on data from different major pairs. Start on dirrefent date within last year
    
How to simulate volume? Volume is not random and have clusters which coresponds to the valotility.. 

Get statistics (run many many times) and measure when in average (in how many candles) the strategy will give the certain profit (0, 10%, 50%)

Will it converge or diverge? 

Question: will we have the different average periods or the same?
If the same - which period will it be.
Can we measure the average speed

What is the properties of random process? Can I measure some properties from stock prices and apply them to the random process to get a 
random_process' with the standats deviations not higher than the measured ones. And then test strategies on that random_process'

"""

"\nTest strategies based on simple technical indicators (RSI, MA, MACD, Bollinger etc.) on\n    - random processes with length of 1000 candles for example \n    - (preferred) (if we think the trading process is not really random walk) on data from different major pairs. Start on dirrefent date within last year\n    \nHow to simulate volume? Volume is not random and have clusters which coresponds to the valotility.. \n\nGet statistics (run many many times) and measure when in average (in how many candles) the strategy will give the certain profit (0, 10%, 50%)\n\nWill it converge or diverge? \n\nQuestion: will we have the different average periods or the same?\nIf the same - which period will it be.\nCan we measure the average speed\n\nWhat is the properties of random process? Can I measure some properties from stock prices and apply them to the random process to get a \nrandom_process' with the standats deviations not higher than the measured ones. And then test strategies on that rand

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ccxt
import math
import datetime
import os

In [2]:
def normalize_remove_time(df_in):
    df_orig = df_in.drop('TIME', axis=1)
    
    # normalize by columns
    df = pd.DataFrame(columns=df_orig.columns.to_list())
    for c in df_orig:
        # minmax
        # df[c]=(df_orig[c] - df_orig[c].mean()) / (df_orig[c].max() - df_orig[c].min())
        # Z-normalization
        df[c]=(df_orig[c] - df_orig[c].mean()) / df_orig[c].std()
    return df


def visualize(data_dict, titel, file_name):
    """
    data_dict: {pair: df}
    """
    pairs_lim = list(data_dict.keys())#[:1]
    # (rows, columns)
    layout = [math.ceil(len(pairs_lim) / 3), 3]
    fig, axes = plt.subplots(nrows=layout[0], ncols=layout[1])#, width_ratios=2, height_ratios=2)
    fig.set_size_inches(w=1760/100, h=layout[0]*640/100)
    fig.suptitle(titel)
    axe = axes.ravel()
    for pair, ax in zip(pairs_lim, axe):
        data_dict[pair].plot(kind='line', title=pair, ax=ax, grid=True)
    # plt.show()
    plt.savefig(f'{file_name}.pdf')


In [54]:
pairs = ["GMX/USDT", 
"BCH/USDT", 
"ETC/USDT", 
"HBAR/USDT", 
"ILV/USDT", 
"TRB/USDT", 
"ETH/USDT", 
"ADA/USDT", 
"NEO/USDT", 
"SAND/USDT", 
"TON/USDT", 
"RUNE/USDT", 
"FIL/USDT", 
"NEAR/USDT", 
"MKR/USDT", 
"LINK/USDT", 
"ATOM/USDT", 
"BSV/USDT", 
"UNI/USDT", 
"LTC/USDT", 
"AAVE/USDT", 
"COMP/USDT", 
"SUSHI/USDT", 
"ZRO/USDT", 
"SOL/USDT", 
"CRV/USDT", 
"BTC/USDT"]
timeframe = "5m"

if timeframe[-1] != "m":
    raise ValueError('timeframe[-1] != "m"')

tf_milliseconds = int(timeframe[:-1]) * 60000

In [48]:
exchange = ccxt.binance()

In [ ]:
tickers = exchange.fetch_tickers()

pairs = []
for symbol, ticker in tickers.items():
    if "/" not in symbol:
        continue
    if "USDT" in symbol:
        pairs.append(symbol)
print(len(pairs))
print(pairs)

In [68]:
p_data = {}
limit=288 # day
# since = exchange.milliseconds () - 86400000  # -1 day from now
since = exchange.milliseconds () - (tf_milliseconds * limit)
for i, pair in enumerate(pairs):
    if i % 100 == 0:
        print(i, len(pairs), pair)
    ohlcv = exchange.fetch_ohlcv(pair, timeframe, since=since, limit=limit)
    p_data[pair] = pd.DataFrame(ohlcv, columns=["TIME", "OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"])
    p_data[pair]["TIME"] = pd.to_datetime(p_data[pair]["TIME"], unit="ms")
print("max: ", p_data[pairs[0]].max(axis=0)['TIME'])
print("min: ", p_data[pairs[0]].min(axis=0)['TIME'])

0 574 BTC/USDT
100 574 ETHBEAR/USDT
200 574 DOTUP/USDT
300 574 AR/USDT
400 574 T/USDT
500 574 PORTAL/USDT
max:  2025-03-05 21:55:00
min:  2025-03-04 22:00:00


In [8]:
"""
correlations inside one pair

get DataFrame with correlations depending on the window from earliest to oldest
"""
pairs_lim = list(p_data.keys())#[:1]
out_pair = {pair: pd.DataFrame(columns=['corr(O-C->V)', 'corr(H-L->V)', 'corr(O-C->H-L)', 'std(O-C)', 'std(H-L)']) for pair in pairs_lim}
for i in range(10, limit):
    for pair in pairs_lim:
        df = normalize_remove_time(p_data[pair])
        df = df.tail(i) # newest are in the tail

        df["OPEN-CLOSE"] = abs(df['OPEN'] - df['CLOSE'])
        df["HIGH-LOW"] = abs(df['HIGH'] - df['LOW'])
        corr = df.corr().drop(["OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"], axis=1)
        out_pair[pair].loc[i] = [corr['OPEN-CLOSE']['VOLUME'], corr['HIGH-LOW']['VOLUME'], corr['OPEN-CLOSE']['HIGH-LOW'], df["OPEN-CLOSE"].std(), df["HIGH-LOW"].std()]

In [ ]:
visualize(out_pair, f'data (y) depending on the window (x) from earliest to oldest limit {limit}', f"10_to_{limit}_newest_to_oldest")

In [9]:
"""
correlations inside one pair

Analyze 'n_periods' periods
Get 'limit' candles from different, consequent, not overlaping period of time in the past starting from earliest
For each period calculate 'corr(O-C->V)', 'corr(H-L->V)', 'corr(O-C->H-L)', 'std(O-C)', 'std(H-L)'
"""
n_periods = 10
limit = 50

pairs_lim = pairs#[:1]

out_pair_1 = {pair: pd.DataFrame(columns=['corr(O-C->V)', 'corr(H-L->V)', 'corr(O-C->H-L)', 'std(O-C)', 'std(H-L)']) for pair in pairs_lim}
for pair in pairs_lim:
    for period in range(1, n_periods+1): # period = from 1 to n_periods
        # since = exchange.milliseconds () - 86400000  # -1 day from now
        since = exchange.milliseconds () - (tf_milliseconds * limit * period)
        ohlcv = exchange.fetch_ohlcv(pair, timeframe, since=since, limit=limit)
        df = normalize_remove_time(pd.DataFrame(ohlcv, columns=["TIME", "OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"]))
    
        df["OPEN-CLOSE"] = abs(df['OPEN'] - df['CLOSE'])
        df["HIGH-LOW"] = abs(df['HIGH'] - df['LOW'])
        corr = df.corr().drop(["OPEN", "HIGH", "LOW", "CLOSE", "VOLUME"], axis=1)
        lst = [corr['OPEN-CLOSE']['VOLUME'], corr['HIGH-LOW']['VOLUME'], corr['OPEN-CLOSE']['HIGH-LOW'], df["OPEN-CLOSE"].std(), df["HIGH-LOW"].std()]
        out_pair_1[pair].loc[period] = lst


In [ ]:
visualize(out_pair_1, f'data (y) depending on the period (x) from earliest to oldest. each x point is calculated by {limit} candles', 
          f"p_{n_periods}_lim_{limit}_newest_to_oldest")

In [6]:
"""
correlations between pairs
"""
import itertools

pairs_lim = list(p_data.keys())#[:1]
combinations = list(itertools.combinations(pairs_lim, 2))
print("combinations: ", len(combinations))

cors = []
for (p1, p2) in combinations:
    df1 = normalize_remove_time(p_data[p1])
    df2 = normalize_remove_time(p_data[p2])
    c = df1['CLOSE'].corr(df2['CLOSE'])
    if not math.isnan(c):
        cors.append([p1, p2, c])

cors.sort(reverse=True, key=lambda x: x[2])

# for (p1, p2, corr) in cors:
#     print(f"{p1} {p2}: \t{corr}")

combinations:  351


In [ ]:
"""
visualize
"""
for_viz = {}
# first 5
for (p1, p2, corr) in cors[:5]:
    for_viz[p1] = normalize_remove_time(p_data[p1])['CLOSE']
    for_viz[p2] = normalize_remove_time(p_data[p2])['CLOSE']
visualize(for_viz, f'first 5 the best correlated: \n{"\n".join([str(c) for c in cors[:5]])}', 
          f"5_best_cor_pairs")


for_viz = {}
# last 5
for (p1, p2, corr) in cors[-5:]:
    for_viz[p1] = normalize_remove_time(p_data[p1])['CLOSE']
    for_viz[p2] = normalize_remove_time(p_data[p2])['CLOSE']
visualize(for_viz, f'last 5 the worst correlated: \n{"\n".join([str(c) for c in cors[-5:]])}', 
          f"5_worst_cor_pairs")

    
# cors[:5]
# cors[-5:]

In [50]:
"""
make time series stationary
"""
from scipy import integrate
import statsmodels.api as sm

pairs_lim = list(p_data.keys())#[:3]

p_data_stationary = {}
for pair in pairs_lim:
    df = normalize_remove_time(p_data[pair])
    # df.Time = pd.to_datetime(df.Time)
    # df = df.set_index('Time')
    df['integrated'] = df['CLOSE'] - df['CLOSE'].shift(1)
    df = df.dropna()
    if df.size > 0:
        p_data_stationary[pair] = df.drop(['OPEN', 'HIGH', 'LOW', 'VOLUME'], axis=1)
        # test stationarity
        # res = sm.tsa.stattools.kpss(p_data_stationary[pair]['integrated'].dropna(), regression='ct')
        # is_stationary = True if res[1] >= 0.1 else False
        # print(f"{pair}: is_stationary: {is_stationary} p value: {res[1]}")


In [ ]:
visualize(p_data_stationary, f'integarted', 
          f"integarted")

In [ ]:
"""
VAR prediction
"""
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.api import VAR

pairs_lim = list(p_data_stationary.keys())[:3]

mdata = pd.DataFrame()
for pair in pairs_lim:
    mdata[f'{pair}'] = p_data_stationary[pair]['integrated']

mdata.reset_index(inplace=True) 
mdata.info()

model = VAR(mdata)
# results = model.fit(2)
model.select_order(2)
results = model.fit(maxlags=2, ic='aic')
lag_order = results.k_ar
results.forecast(mdata.values[-lag_order:], 5)
results.plot_forecast(10)



In [ ]:
def visualize_line_by_line(data_dict):
    """
    data_dict: {pair: df}
    """
    pairs_lim = list(data_dict.keys())
    # (rows, columns)
    layout = [math.ceil(len(pairs_lim) / 1), 1]
    fig, axes = plt.subplots(nrows=layout[0], ncols=layout[1])
    fig.set_size_inches(w=1760/100, h=layout[0]*140/100)
    # fig.suptitle(titel)
    fig.subplots_adjust(bottom=0, top=1, left=0, right=1, wspace=0, hspace=0)
    axe = axes.ravel()
    
    # Major ticks every 20, minor ticks every 5
    # major_ticks = np.arange(0, 101, 20)

    
    for pair, ax in zip(pairs_lim, axe):
        # ax.get_xaxis().set_visible(False)
        ax.set_xticklabels([])
        # ax.set_yticklabels([])
        ax.tick_params(left=False, bottom=False)
       
        # ax.set_xticks(major_ticks)
        ax.set_xticks(np.arange(0, data_dict[pair].size, 5))#, minor=True)

        
        data_dict[pair].plot(kind='line', title=pair, ax=ax, grid=True)

    # plt.subplots_adjust(wspace=0, hspace=0)
    # plt.show()
    plt.savefig(f'574_pairs.pdf')

_data = {}
for pair in p_data.keys():
    # _data[pair] = p_data_stationary[pair]#['integrated']
    if p_data[pair]['CLOSE'].size == 0:
        continue
    _data[pair] = p_data[pair]['CLOSE']


visualize_line_by_line(_data)

# _data = {}
# for pair in p_data_stationary.keys():
#     _data[pair] = p_data_stationary[pair]['integrated']
#     # _data[pair] = p_data_stationary[pair]['CLOSE']


# visualize_line_by_line(_data)


In [78]:
"""
arbitrage between stocks
"""
timeframe = "5m"

if timeframe[-1] != "m":
    raise ValueError('timeframe[-1] != "m"')

tf_milliseconds = int(timeframe[:-1]) * 60000

exchanges = {
"binance": ccxt.binance(),
"bybit": ccxt.bybit(),
"htx": ccxt.htx(),
# "hyperliquid": ccxt.hyperliquid(), # DEX
"mexc": ccxt.mexc(),
"okx": ccxt.okx(),
"kucoin": ccxt.kucoin(),
}


# tickers = exchange.fetch_tickers()

# pairs = []
# for symbol, ticker in tickers.items():
#     if "/" not in symbol:
#         continue
#     if "USDT" in symbol:
#         pairs.append(symbol)

for ex_name in exchanges:
    price = exchanges[ex_name].fetch_ticker('BTC/USDT')['last']
    print(ex_name, price)

binance 90127.77
bybit 90126.32
htx 90124.79
mexc 90127.76
okx 90130.0
kucoin 90125.6
